# 21 — Rust controller bakeoff (no rollout)

Compare five **oracle-shelf** ladder controllers on shared seeds (SIM-01=B):

| Arm | Description |
| --- | --- |
| `constant` | Fixed case-rounded protection order |
| `rung0` | Corrected age-blind ladder |
| `sw` | Damped survival-weighted |
| `sla_pb` | Window SLA Poisson–binomial fast path |
| `sla_mc` | Window SLA Monte Carlo oracle |

**Excluded:** `rollout`, `dp`.

Default belief world is **oracle** (`evaluate_alpha_episode_outcomes`). Set
`BELIEF_WORLD=filtered` for an optional appendix using `session.act` on P0 channels
(four arms — `rung0` has no session mapping).

Prelim budget: `n_burn=2`, `n_score=14`, 4 seeds → **20 Modal shards** (<10 min wall).
Set `SMOKE=True` for one seed / one arm plumbing.

In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path
from typing import Literal

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd().resolve()
for _candidate in [REPO_ROOT, *REPO_ROOT.parents]:
    if (_candidate / "src" / "blueberries_voi").is_dir():
        REPO_ROOT = _candidate
        break

_wheel_dir = REPO_ROOT / "dist" / "wheel"
os.environ.setdefault("BLUEBERRIES_VOI_BACKEND", "rust")
if _wheel_dir.is_dir():
    os.environ["BLUEBERRIES_VOI_WHEEL"] = str(_wheel_dir)

from blueberries_voi.experiments.controller_bakeoff import (
    ARM_LABELS,
    BAKEOFF_ARMS,
    DEFAULT_CONTROLLER_SEEDS,
    arms_for_belief_world,
    belief_world_from_env,
)
from blueberries_voi.experiments.modal_dispatch import run_batch
from blueberries_voi.experiments.policy_bakeoff_viz import (
    write_paired_delta_figure,
    write_profit_bars_figure,
    write_runtime_bars_figure,
    write_runtime_frontier_figure,
    write_waste_stockout_bars_figure,
)
from blueberries_voi.experiments.voi_profit import load_damped_sw_bo_params

BATCH_MODE: Literal["modal", "local"] = "local"
SMOKE = False
BELIEF_WORLD = belief_world_from_env()
ARMS = arms_for_belief_world(BELIEF_WORLD)
SEEDS = (42,) if SMOKE else DEFAULT_CONTROLLER_SEEDS
N_BURN, N_SCORE = 2, 14
RHO = load_damped_sw_bo_params()[1]
OUT_DIR = Path("notebooks/outputs/nb21_controller_bakeoff")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(
    f"BATCH_MODE={BATCH_MODE} SMOKE={SMOKE} belief={BELIEF_WORLD} "
    f"seeds={SEEDS} arms={ARMS} rho={RHO:.2f}"
)

## Runtime probe (local)

Before Modal, we benchmarked `sla_mc` vs `sw` at `n_burn=2`, `n_score=14` on one seed:

| Arm | Wall (s) |
| --- | --- |
| sw | ~0.03 |
| sla_pb | ~0.03 |
| sla_mc | ~0.15 |

All arms stay well under the ~2–3 min/shard ceiling — **no `sla_mc_n_score` reduction**.

In [ ]:
rows = run_batch(
    "controller_bakeoff",
    BATCH_MODE,
    smoke=SMOKE,
    seeds=SEEDS,
    arms=ARMS,
    rho=RHO,
    belief_world=BELIEF_WORLD,
    n_burn=N_BURN,
    n_score=N_SCORE,
    out_path=OUT_DIR / "controller_bakeoff_rows.json",
)
df = pd.DataFrame(rows)
df["arm_label"] = df["arm_id"].map(ARM_LABELS)
display(df.groupby("arm_id")[["profit", "waste", "stockout", "elapsed_s"]].mean())

## Controller differences — profit / waste / stockout

In [ ]:
shard_rows = json.loads((OUT_DIR / "controller_bakeoff_rows.json").read_text())
if "elapsed_s" not in df.columns and shard_rows:
    df = pd.DataFrame(shard_rows)

write_profit_bars_figure(OUT_DIR / "01_profit_by_arm.png", shard_rows)
write_waste_stockout_bars_figure(OUT_DIR / "02_waste_stockout.png", shard_rows)
write_runtime_bars_figure(OUT_DIR / "03_runtime_by_arm.png", shard_rows)
write_paired_delta_figure(OUT_DIR / "04_paired_delta_vs_sw.png", shard_rows, baseline="sw")

fig, ax = plt.subplots(figsize=(7, 4))
for arm in ARMS:
    sub = df[df["arm_id"] == arm]
    ax.scatter(sub["elapsed_s"], sub["profit"], label=ARM_LABELS.get(arm, arm), s=60)
ax.set_xlabel("Shard wall time (s)")
ax.set_ylabel("Profit")
ax.set_title("Runtime vs profit frontier")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(OUT_DIR / "05_runtime_frontier.png", dpi=120)
plt.close(fig)

frontier = [
    {
        "arm_idx": float(i),
        "profit": float(r["profit"]),
        "seconds": float(r.get("elapsed_s", 0.0)),
        "fill_proxy": 1.0 - float(r["stockout"]) / max(1, N_SCORE * 20),
    }
    for i, r in enumerate(shard_rows)
]
write_runtime_frontier_figure(OUT_DIR / "runtime_frontier.json", frontier)
print(f"Wrote figures under {OUT_DIR}")

## Per-seed runtime table

In [ ]:
runtime_pivot = df.pivot_table(
    index="seed", columns="arm_id", values="elapsed_s", aggfunc="first"
)
display(runtime_pivot.round(3))
runtime_pivot.to_csv(OUT_DIR / "runtime_by_seed.csv")

## Optional appendix — filtered beliefs (`BELIEF_WORLD=filtered`)

Re-run with `os.environ["BELIEF_WORLD"] = "filtered"` and `BATCH_MODE="local"` or Modal.
Grid drops `rung0` (no `session.act` mapping).

In [ ]:
if BELIEF_WORLD == "filtered":
    write_paired_delta_figure(
        OUT_DIR / "06_filtered_paired_delta_vs_sw.png",
        shard_rows,
        baseline="sw",
        title="Filtered belief: profit delta vs sw",
    )
else:
    print("Skipping filtered-only chart (oracle run).")